# AAROH Transformer Distress Model

This notebook is the supported training entry point for the multilingual transformer-based Distress model. Labels are synthetic demonstration labels, not clinical ground truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to the repository directory after cloning or uploading AAROH.
REPO_DIR = '/content/AAROH'
import os
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Expected the AAROH repository at {REPO_DIR}')
os.chdir(REPO_DIR)

In [ ]:
%pip install -q torch transformers scikit-learn
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AAROH')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'distress'
OUTPUT_DIR = Path('models/distress')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint directory:', CHECKPOINT_DIR)
print('Export directory:', OUTPUT_DIR)

In [ ]:
!python -m backend.ml.training.train_distress \
  --data-dir datasets/processed \
  --output-dir models/distress \
  --checkpoint-dir checkpoints/distress \
  --drive-checkpoint-dir $CHECKPOINT_DIR \
  --execution-mode PYTORCH_FROZEN \
  --model-name distilbert-base-multilingual-cased \
  --batch-size 16 \
  --epochs 3 \
  --gradient-accumulation-steps 2 \
  --fp16

In [ ]:
!python -m backend.ml.training.evaluate_distress_model \
  --model-dir models/distress \
  --output-file models/distress/metrics.json \
  --sample-count 80

In [ ]:
from pathlib import Path
required = [
    Path('models/distress/config.json'),
    Path('models/distress/metadata.json'),
    Path('models/distress/pytorch_model.bin'),
    Path('models/distress/tokenizer.json'),
    Path('models/distress/tokenizer_config.json'),
    Path('models/distress/label_mapping.json'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing exported Distress artifacts: {missing}')
print('Transformer Distress artifacts exported:', [str(path) for path in required])